In [ ]:
import pandas as pd
from IPython.display import Markdown, display

dataset_names = ["ml_d1_predelivery", "ml_d2_earlydeath", "ml_d3_latedeath"]


interpretation_usage = {
    "ml_d1_predelivery": {
        "Outcome Death": ("Yes", "Target renamed to Death"),
        "Maternal Age": ("Yes", ""),
        "School Level": ("Yes", "Encoded numerically"),
        "Years of Education": ("Yes", ""),
        "Parity": ("Yes", ""),
        "Maternal Height": ("No", "Used only to derive BMI"),
        "Maternal Weight": ("No", "Used only to derive BMI"),
        "Multiple Birth": ("Yes", ""),
        "Antenatal Visits": ("Yes", ""),
        "Gestation": ("Yes", ""),
        "Type of Delivery Place": ("No", "Dropped before interpretation"),
        "Delivery Place": ("Yes", "Encoded as Delivery Place Home"),
        "Mode of Delivery": ("Yes", "Encoded as Mode of Delivery Cesarea"),
        "Pre-term Delivery": ("Yes", "Encoded as binary"),
    },
    "ml_d2_earlydeath": {
        "Early Neonatal Death": ("Yes", "Target renamed to Death"),
        "Maternal Age": ("Yes", ""),
        "School Level": ("Yes", "Encoded numerically"),
        "Years of Education": ("Yes", ""),
        "Parity": ("Yes", ""),
        "Maternal Height": ("No", "Used only to derive BMI"),
        "Maternal Weight": ("No", "Used only to derive BMI"),
        "Multiple Birth": ("Yes", ""),
        "Antenatal Visits": ("Yes", ""),
        "Gestation": ("Yes", ""),
        "Type of Delivery Place": ("No", "Dropped before interpretation"),
        "Delivery Place": ("Yes", "Encoded as Delivery Place Home"),
        "Birthweight": ("Yes", ""),
        "Birthweight Measure": ("No", "Dropped before interpretation"),
        "Mode of Delivery": ("Yes", "Encoded as Mode of Delivery Cesarea"),
        "Baby Sex": ("Yes", "Encoded as Baby Sex Female"),
        "Dexamethasone": ("Yes", ""),
        "CPAP": ("Yes", ""),
        "Oxygen": ("Yes", ""),
        "Kangaroo Mother Care": ("Yes", ""),
        "Cord care Chlorhexidine": ("Yes", ""),
        "Bag and Mask Resuscitation": ("Yes", ""),
        "Pre-term Delivery": ("Yes", "Encoded as binary"),
    },
    "ml_d3_latedeath": {
        "Late Neonatal Death": ("Yes", "Target renamed to Death"),
        "Maternal Age": ("Yes", ""),
        "School Level": ("Yes", "Encoded numerically"),
        "Years of Education": ("Yes", ""),
        "Parity": ("Yes", ""),
        "Maternal Height": ("No", "Used only to derive BMI"),
        "Maternal Weight": ("No", "Used only to derive BMI"),
        "Multiple Birth": ("Yes", ""),
        "Antenatal Visits": ("Yes", ""),
        "Gestation": ("Yes", ""),
        "Type of Delivery Place": ("No", "Dropped before interpretation"),
        "Delivery Place": ("Yes", "Encoded as Delivery Place Home"),
        "Birthweight": ("Yes", ""),
        "Birthweight Measure": ("No", "Dropped before interpretation"),
        "Mode of Delivery": ("Yes", "Encoded as Mode of Delivery Cesarea"),
        "Baby Sex": ("Yes", "Encoded as Baby Sex Female"),
        "Dexamethasone": ("Yes", ""),
        "CPAP": ("Yes", ""),
        "Oxygen": ("Yes", ""),
        "Kangaroo Mother Care": ("Yes", ""),
        "Cord care Chlorhexidine": ("Yes", ""),
        "Bag and Mask Resuscitation": ("Yes", ""),
        "Pre-term Delivery": ("Yes", "Encoded as binary"),
    },
}


def add_common_column_stats(summary, df, dataset_name):
    summary.index.name = "column"
    usage_pairs = [interpretation_usage[dataset_name].get(col, ("Unknown", "")) for col in summary.index]
    summary["dtype"] = df[summary.index].dtypes.astype(str)
    summary["used in interpretation"] = [pair[0] for pair in usage_pairs]
    summary["interpretation note"] = [pair[1] for pair in usage_pairs]
    summary["missing_n"] = df[summary.index].isna().sum()
    summary["missing_pct"] = (df[summary.index].isna().mean() * 100).round(2)
    summary["non_missing_n"] = df[summary.index].notna().sum()
    summary["unique_n"] = df[summary.index].nunique(dropna=True)
    return summary


def describe_numeric_for_ml(df, dataset_name):
    numeric_df = df.select_dtypes(include="number")
    if numeric_df.empty:
        return pd.DataFrame()

    summary = numeric_df.describe().T
    summary = add_common_column_stats(summary, df, dataset_name)
    ordered_columns = [
        "dtype",
        "non_missing_n",
        "missing_n",
        "missing_pct",
        "unique_n",
        "count",
        "mean",
        "std",
        "min",
        "25%",
        "50%",
        "75%",
        "max",
        "used in interpretation",
        "interpretation note",
    ]
    return summary.reindex(columns=ordered_columns)


def describe_categorical_for_ml(df, dataset_name):
    categorical_df = df.select_dtypes(exclude="number")
    if categorical_df.empty:
        return pd.DataFrame()

    summary = categorical_df.describe().T
    summary = add_common_column_stats(summary, df, dataset_name)
    ordered_columns = [
        "dtype",
        "non_missing_n",
        "missing_n",
        "missing_pct",
        "unique_n",
        "count",
        "top",
        "freq",
        "used in interpretation",
        "interpretation note",
    ]
    return summary.reindex(columns=ordered_columns)


for dataset_name in dataset_names:
    df = pd.read_csv(f"../data/processed/{dataset_name}.csv")
    numeric_summary = describe_numeric_for_ml(df, dataset_name)
    categorical_summary = describe_categorical_for_ml(df, dataset_name)

    display(Markdown(f"## {dataset_name}"))
    display(Markdown("### Numeric variables"))
    display(numeric_summary)
    display(Markdown("### Categorical variables"))
    display(categorical_summary)
